# PopOut Project

## Summary

This project was developed for the Artificial Inteligence course during the first semester of 2026 in the University of Porto. Our main goal was applying coding and critical thinking skills to adapt adversary search algorithms (Monte Carlo Tree Search) and learning algorithms (ID3 Decision Trees). Our project is divided mainly in three parts:

- Creating the game class that includes the game logic (board, moves, rules) and the main loop.

- Implementing the Monte Carlo Tree Search algorithm with the UCT formula for better exploration/exploitation balance.

- Defining the ID3 Decision Tree, its parameters and splits, and training it on existing Iris data and later on generated PopOut game data.

### PopOut - Game Class and loop

PopOut is a variant of the game Connect4. The main goal is to land 4 consecutive pieces of their own colour on a horizontal, vertical or diagonal line. The difference between this variant and the original game consists in a new move, that allows the player to "pop" the piece from the bottom row of the board, if it's one's own colour. This creates new possibilities and increases the game complexity. It also creates a small issue with determining the winner of the game, as by popping out, rows of four pieces can be made for both players simultaneously. This and the following are to be treated on the game loop:

- If a pop move creates wins for both players, the one who made the move wins the game.

- In the case of a full board the current player is allowed to end the game as a draw instead of making a pop move.

- If the board state is repeated 3 times, the game ends as a draw.

#### Board Structure

Instead of a regular matrix, we looked for a better, more optimized format and we reached the conclusion that using a bitboard was the most efficient choice.
Therefore the board is defined as a 49 long integer (49 instead of 42 because we have a buffer row that separates the columns on the bitboard), that allows faster and lower cost operations, using shifts, additions, and logic operations such as "OR", "AND", "XOR", "NOT" to combine the board states and update the game. Alongside this measure, we allocated every board state and its mask (that is used in a "XOR" operation to differentiate each player's pieces) an identifier to create a HashMap. This data structure allows for an easier draw identification and facilitates our MCTS algorithm to access previously generated states.

In [ ]:
from random_play import RandomPlay
from mcts_play import MCTSPlay
from id3_play import ID3Play

class Connect4:
    """
    Main class to manage the game initialization
    """
    def __init__(self):
        """
        Initializes the Connect4 game.
        """
        self.turn = 0
        self.result = None
        self.terminal = False
        
    def get_initial_position(self):
        """
        Gets the starting position of the game.

        Returns:
            Position: The initial board state with turn 0.
        """
        return Position(self.turn)
                
class Position:
    """
    Represents a state of the game board using bitboards.
    """
    # Removed 'history' parameter to avoid memory overload during rollouts
    def __init__(self, turn, mask = 0, position = 0, num_turns = 0):
        """
        Initializes a game state.

        Args:
            turn (int): The current player's turn (0 or 1).
            mask (int): Bitboard representing all pieces on the board. Defaults to 0.
            position (int): Bitboard representing the current player's pieces. Defaults to 0.
            num_turns (int): Counter for the number of turns played. Defaults to 0.
        """
        self.turn = turn
        self.result = None
        self.terminal = False
        self.num_turns = num_turns
        self.mask = mask
        self.position = position
        self._compute_hash()
                
    def move(self, loc):
        """
        Applies a move and returns a new board state.

        Args:
            loc (int): The chosen move. 0-6 for dropping a piece, 7-13 for popping a piece from the bottom, or -1 for a draw.

        Returns:
            Position: A new Position instance of the board after the move.
        """
        # Rules 2 and 3, draw
        if loc == -1:
            # Creation without history copy (pure math and bitwise operations)
            new_pos = Position(int(not self.turn), self.mask, self.position ^ self.mask, self.num_turns + 1)
            new_pos.terminal = True
            new_pos.result = 0
            return new_pos

        is_pop = False
        
        if 0 <= loc <= 6:
            # Normal move (0-6)
            new_position = self.position ^ self.mask
            new_mask = self.mask | (self.mask + (1 << (loc*7)))
        else:
            # Pop move (7-13)
            c = loc - 7
            col_mask = 0b111111 << (c * 7)
            
            curr_col = self.position & col_mask
            opp_col = (self.position ^ self.mask) & col_mask
            
            # shift move
            new_curr_col = (curr_col >> 1) & col_mask
            new_opp_col = (opp_col >> 1) & col_mask
            
            new_curr = (self.position & ~col_mask) | new_curr_col
            new_opp = ((self.position ^ self.mask) & ~col_mask) | new_opp_col
            
            new_position = new_opp
            new_mask = new_curr | new_opp
            is_pop = True

        # mathematical instance creation (tries to fix garbage collection bottleneck)
        new_pos = Position(int(not self.turn), new_mask, new_position, self.num_turns + 1)
        new_pos.game_over(is_pop)
        return new_pos
    
    # return list of legal moves
    def legal_moves(self):
        """
        Determines all valid moves for the current board state.

        Returns:
            list: A list of integers representing valid moves (0-6 for drops, 7-13 for pops, -1 for draw).
        """
        bit_moves = []
        # regular moves
        for i in range(7):
            col_mask = 0b111111 << 7 * i
            if col_mask != self.mask & col_mask:
                bit_moves.append(i)
                
        # pop move
        # checks if the piece belongs to the current player
        for i in range(7):
            bottom_bit = 1 << (i * 7)
            if self.position & bottom_bit:
                bit_moves.append(i + 7)

        # draw check, full board
        is_full = (self.mask == 279258638311359)
        
        if is_full:
            bit_moves.append(-1)
            
        return bit_moves

    def game_over(self, is_pop=False):
        """
        Checks if the game has ended and updates the terminal status and result.

        Args:
            is_pop (bool, optional): Indicates if the last move was a PopOut move. Defaults to False.
        """
        # sets result to -1, 0, or 1 if game is over (otherwise self.result is None)
        just_moved_won = self.connected_four_fast()
        about_to_move_won = False
        
        if is_pop:
            about_to_move_won = self.connected_four_fast(self.position)
            
        if just_moved_won or about_to_move_won:
            self.terminal = True
            
            if just_moved_won:
                self.result = 1 if self.turn == 1 else -1 
            else:
                self.result = 1 if self.turn == 0 else -1
        else:
            self.terminal = False
            self.result = None
            
        if not self.terminal and not self.legal_moves():
            self.terminal = True
            self.result = 0
            
    def connected_four_fast(self, board_position=None):
        """
        Detection of four connected pieces using bitwise shifts.

        Args:
            board_position (int): The board to evaluate. If None, evaluates the current player's board.

        Returns:
            bool: True if there is a 4-in-a-row, False otherwise.
        """
        if board_position is None:
            board_position = self.position ^ self.mask
            
        # Horizontal check
        m = board_position & (board_position >> 7)
        if m & (m >> 14):
            return True
        # Diagonal \
        m = board_position & (board_position >> 6)
        if m & (m >> 12):
            return True
        # Diagonal /
        m = board_position & (board_position >> 8)
        if m & (m >> 16):
            return True
        # Vertical
        m = board_position & (board_position >> 1)
        if m & (m >> 2):
            return True
        # Nothing found
        return False
    
    
    def _compute_hash(self):
        """
        Computes a unique hash for the current board state.
        """
        position_1 = self.position if self.turn == 0 else self.position ^ self.mask
        self.hash = 2 * hash((position_1, self.mask)) + self.turn
    
    def __hash__(self):
        return self.hash
        
    def __eq__(self, other):
        return isinstance(other, Position) and self.turn == other.turn and self.mask == other.mask and self.position == other.position

    # Visual interface
    def print_board(self, legal_moves=None):
        """
        Displays the board to the console.

        Args:
            legal_moves (list): List of legal moves to be displayed for the player.
        """
        p0_bits = self.position if self.turn == 0 else (self.position ^ self.mask)
        p1_bits = self.position if self.turn == 1 else (self.position ^ self.mask)
        
        display = "\n  7  8  9 10 11 12 13 : Pop"
        display += "\n  0  1  2  3  4  5  6 : Drop\n-----------------------"
        for row in range(5, -1, -1):
            line = "|"
            for col in range(7):
                bit = 1 << (col * 7 + row)
                if p0_bits & bit:
                    line += " O " # Player 0
                elif p1_bits & bit:
                    line += " X " # Player 1
                else:
                    line += " . " # Empty
            display += "\n" + line + "|"
        display += "\n-----------------------"

        if legal_moves is not None:
                    drops = [m for m in legal_moves if 0 <= m <= 6]
                    pops = [m for m in legal_moves if 7 <= m <= 13]
                    draw = [-1] if -1 in legal_moves else []
                    
                    display += "\nLegal Moves:"
                    if drops: display += f"\n  Drops: {drops}"
                    if pops:  display += f"\n  Pops:  {pops}"
                    if draw:  display += f"\n  Draw:  [-1]"
                    display += "\n"

        print(display)

# Game loop
if __name__ == "__main__":
    
    def choose_player(num_player):
        """
        Displays a menu for the user to select the type of agent for a specific player.

        Args:
            num_player (int): The player ID (0 or 1).

        Returns:
            object: An instance of the selected player class (HumanPlay, RandomPlay, MCTSPlay, or ID3Play).
        """
        while True:
            print(f"\nChoose a player for {num_player}")
            print("1 - Human")
            print("2 - Random Play")
            print("3 - MCTS")
            print("4 - ID3 (MCTS)")
            player_str = input("Enter 1 or 2 or 3 or 4: ").strip()
            
            if player_str == '1':
                return HumanPlay(f"Player {num_player} (Human)")
            elif player_str == '2':
                return RandomPlay(f"Player {num_player} (Random)")
            elif player_str == '3':
                return MCTSPlay(f"Player {num_player} (MCTS)", time_limit=2.0)
            elif player_str == '4':
                return ID3Play(f"Player {num_player} (ID3)")
            else:
                print("Invalid player! Please enter 1, 2 or 3.")

    agents = {
        0: choose_player(0),
        1: choose_player(1)
    }

    game = Connect4()
    pos = game.get_initial_position()
    
    print(f"\n--- Game start: {agents[0].name} VS {agents[1].name} ---")

    # memory stored to check for draws
    global_history = {}

    # Main Loop
    while not pos.terminal:
        # updates the current board state occurences
        global_history[pos.hash] = global_history.get(pos.hash, 0) + 1
        
        legal = pos.legal_moves()
        
        # Injects the draw option (-1) if threefold repetition is met
        if global_history[pos.hash] >= 3 and -1 not in legal:
            legal.append(-1)
            print("\n[!] Attention: This state has repeated 3 times. Draw (-1) is available.")

        pos.print_board(legal_moves=legal)
        
        current_agent = agents[pos.turn]
        print(f"\n{current_agent.name}'s turn.")
        
        move = current_agent.get_move(pos)
        pos = pos.move(move)

    # End of game
    pos.print_board()
    if pos.result == 0:
        print("It's a Draw!")
    else:
        winner = 0 if pos.result == 1 else 1
        print(f"Player {winner} wins!")


Choose a player for 0
1 - Human
2 - Random Play
3 - MCTS
4 - ID3 (MCTS)


## Human input collection

A simple prompt to read human player's moves.

In [ ]:
class HumanPlay:
    """
    Wrapper class that manages the human player input.

    Attributes:
        name (str): The display name of the human player.
    """
    def __init__(self, name="Human"):
        """
        Initializes the instance
        """
        self.name = name

    def get_move(self, position):
        """
        Prompts the user for a move and validates it against the legal moves.

        Args:
            position : The current state of the game board.

        Returns:
            int: The index of the selected valid move.
        """
        legal = position.legal_moves()
        while True:
            try:
                move = int(input(f"[{self.name}] Choose a move : "))
                if move in legal:
                    return move
                print("Invalid move! Try again.")
            except ValueError:
                print("Please enter a number.")

## Random play

Play's a random move from the legal action list. It's slowed down for better readability for the user.

In [ ]:
import random
import time

class RandomPlay:
    """
    Wrapper class that manages an agent that plays completely random moves.

    Attributes:
        name (str): The display name of the random agent.
    """
    def __init__(self, name="Random"):
        """
        Initializes the RandomPlay instance.

        Args:
            name (str, optional): The name of the agent. Defaults to "Random".
        """
        self.name = name

    def get_move(self, position):
        """
        Evaluates the board state and returns a random legal move.

        Args:
            position (Position): The current state of the game board.

        Returns:
            int: The index of the randomly selected valid move.
        """
        legal_moves = position.legal_moves()
        
        time.sleep(0.5) # Doesn't let the game play too fast 
        
        move = random.choice(legal_moves)
        print(f"{self.name} played: {move}")
        return move

## Monte Carlo Tree Search

In [4]:
import random
import math
import time
import os
from multiprocessing import Pool

def mcts_worker(args):
    """
    Worker function to build an MCTS tree in a separate CPU core.

    Args: tuple containing root_state and time_limit.

    Return: tuple containing children_stats (dict), iterations (int) and total rollouts (int).
    """
    root_state, time_limit = args
    
    # Ensures each CPU core generates completely different random games
    random.seed(os.urandom(4))

    nodes, iterations, total_rollouts = build_mcts_tree(root_state, time_limit)
    
    children_stats = {}
    for loc in root_state.legal_moves():
        next_state = root_state.move(loc)
        if next_state in nodes:
            w, n, _ = nodes[next_state]
            children_stats[loc] = (w, n)
            
    return children_stats, iterations, total_rollouts


def build_mcts_tree(root_state, time_limit):
    """
    Builds the MCTS statistical tree given a time limit.

    Args: tuple containgin root_state and time_limit.

    Return: tuple containing nodes (dict), iterations (int) and total_rollouts (int).
    """
    # Structure: {state: (total_reward, visits, {parent_state: parent_visits})}
    nodes = {} 
    nodes[root_state] = (0.0, 0.0, {root_state: 0})
    
    start_time = time.time()
    iterations = 0
    total_rollouts = 0 
    
    while time.time() - start_time < time_limit:   
        iterations += 1

        # ==========================================
        # SELECTION
        # ==========================================
        selection_path = selection_phase(nodes, root_state)
        selected_node = selection_path[-1]
        
        _, visits, _ = nodes[selected_node]
        
        # ==========================================
        # EXPANSION
        # ==========================================
        if visits > 0 and not selected_node.terminal:
            legal_moves = selected_node.legal_moves()
            for loc in legal_moves:
                new_state = selected_node.move(loc)
                if new_state not in nodes:
                    nodes[new_state] = (0.0, 0.0, {selected_node: 0})
                    
            loc = random.choice(legal_moves)
            child_state = selected_node.move(loc)
            
            # ==========================================
            # SIMULATION / ROLLOUT
            # ==========================================
            reward = 0
            num_runs = 10
            for _ in range(num_runs):
                reward += simulate_rollout(child_state)
            total_rollouts += num_runs
            
            w, n, parent_n_dict = nodes[child_state]
            if selected_node not in parent_n_dict:
                parent_n_dict[selected_node] = 0
            parent_n_dict[selected_node] += 1
            nodes[child_state] = (w + reward, n + num_runs, parent_n_dict)

        else:
            # ==========================================
            # SIMULATION / ROLLOUT (Direct Simulation)
            # ==========================================
            reward = 0
            num_runs = 10
            for _ in range(num_runs):
                reward += simulate_rollout(selected_node)   
            total_rollouts += num_runs
        
        # ==========================================
        # BACKPROPAGATION
        # ==========================================
        parent = root_state
        for state_in_path in selection_path:
            w, n, parent_n_dict = nodes[state_in_path]
            parent_n_dict[parent] += num_runs
            nodes[state_in_path] = (w + reward, n + num_runs, parent_n_dict)
            parent = state_in_path
            
    return nodes, iterations, total_rollouts


def mcts_agent(time_limit, num_workers=4):
    """
    Wraps the MCTS strategy, returning a move function.

    Args: tuple containing time_limit and num_workers.

    Return: function that takes current_state and returns the optimal move.
    """
    def strat(current_state):
        # Prepare arguments for multiprocessing
        args = [(current_state, time_limit) for _ in range(num_workers)]
        
        # Run MCTS in parallel across available CPU cores
        with Pool(processes=num_workers) as pool:
            results = pool.map(mcts_worker, args)
            
        total_iters = 0
        total_rollouts = 0
        merged_stats = {}
        
        # Merge stats from all workers
        for children_stats, iters, rollouts in results:
            total_iters += iters
            total_rollouts += rollouts
            
            for loc, (w, n) in children_stats.items():
                if loc not in merged_stats:
                    merged_stats[loc] = [0.0, 0.0]
                merged_stats[loc][0] += w
                merged_stats[loc][1] += n

        iter_per_sec = total_iters / time_limit
        rollouts_per_sec = total_rollouts / time_limit
        
        print(f"Number of MCTS iterations: {total_iters} ({iter_per_sec:.0f} iters/sec) [Cores: {num_workers}]")
        print(f"Number of MCTS rollouts:   {total_rollouts} ({rollouts_per_sec:.0f} rollouts/sec) [Cores: {num_workers}]")
        
        player = current_state.turn
        best_score = float('-inf') if player == 0 else float('inf')
        next_best_move = None
   
        for loc in current_state.legal_moves():
            if loc not in merged_stats:
                score = 0.0
            else:
                w, n = merged_stats[loc]
                score = 0.0 if n == 0 else w / n
            
            if score < best_score and player == 1: # Player 1 minimizes
                best_score = score
                next_best_move = loc
            elif score > best_score and player == 0: # Player 0 maximizes
                best_score = score
                next_best_move = loc
                
        # Safety fallback
        if next_best_move is None:
            return random.choice(current_state.legal_moves())
            
        return next_best_move
    return strat
        

def simulate_rollout(state, max_depth=50):
    """
    Executes random rollout from given state until the game ends or reaches the number of plays determined by max_depoth.

    Args: tuple containing state and max_depth.

    Return: float for standardized reward: 1.0 for player 0 win, 0.0 for player 1 win, and 0.5 for a draw.
    """
    current_state = state
    depth = 0
    
    while not current_state.terminal and depth < max_depth:
        moves = current_state.legal_moves()
        loc = random.choice(moves)
        current_state = current_state.move(loc) 
        depth += 1
        
    if not current_state.terminal:
        return 0.5 

    if current_state.result == 1: return 1.0
    elif current_state.result == -1: return 0.0
    return 0.5
        

def selection_phase(nodes, root_state):
    """
    Navigates the tree using the UCB1 policy.

    Args: tuple containing nodes and root_state.

    Return: a list containing the path of traversed nodes from the root to the selected leaf.
    """
    current_node = root_state
    path = []
    visited = set() # Cycle detector for PopOut infinite loops
    
    while True:
        if current_node in visited:
            return path
        
        visited.add(current_node)
        
        _, visits, _ = nodes[current_node]
        path.append(current_node)
        
        if visits == 0:
            return path
        
        legal_moves = current_node.legal_moves()
        next_player = current_node.turn
        
        best_score = float('-inf') if next_player == 0 else float('inf')
        next_best_node = None
        
        for loc in legal_moves:
            result_state = current_node.move(loc)
            
            if result_state not in nodes:
                return path
                
            temp_w, temp_visits, temp_parent_n_count = nodes[result_state]
            if current_node not in temp_parent_n_count:
                temp_parent_n_count[current_node] = 0
                
            if temp_parent_n_count[current_node] == 0:
                path.append(result_state)
                return path
                        
            score = calculate_ucb_score(nodes[current_node][1], temp_parent_n_count[current_node], temp_w / temp_visits, next_player)
            
            if score < best_score and next_player == 1:
                best_score = score
                next_best_node = result_state
            elif score > best_score and next_player == 0:
                best_score = score
                next_best_node = result_state
                
        current_node = next_best_node
        if current_node is None:
            return path
            
            
def calculate_ucb_score(parent_visits, node_visits, win_rate, player, exploration_const=2.0):
    """
    Upper Confidence Bound (UCB1) formula.

    Args: tuple containg parent_visits, node_visits, win_rate, player and exploration_const

    Return: a float of the calculated UCB1 value
    """
    exploration_term = math.sqrt(exploration_const * math.log(parent_visits) / node_visits)
    if player == 0: 
        return win_rate + exploration_term
    return win_rate - exploration_term


class MCTSPlay:
    """
    Wrapper class that manages the MCTS player.

    Atributtes:
        name (str): the display name of the agent.
        time_limit (float): maximum calculation time per move.
        num_workers (int): number of parallel processes to use.
        strategy (function): the strategic move evaluation function.
    """
    def __init__(self, name="MCTS", time_limit=2.0, num_workers=4):
        """Initializes the MCTSPlay instance with custom settings.

        Args: tuple containing name, time_limit and num_workers.
        """
        self.name = name
        self.time_limit = time_limit
        self.num_workers = num_workers
        self.strategy = mcts_agent(self.time_limit, self.num_workers)

    def get_move(self, position):
        """
        Evaluates the board state and returns the optimal move.
        
        Args: position.

        Return: the index of the selected play (int)
        """
        print(f"[{self.name}] Thinking for {self.time_limit} seconds using {self.num_workers} cores...")
        move = self.strategy(position)
        print(f"{self.name} chose to play: {move}")
        return move

## ID3 Decision Tree

A ID3 é um algoritmo de aprendizagem supervisionada que constrói uma árvore de decisão a partir de dados históricos.

O algoritmo analisa o dataset gerado pelo MCTS e utiliza o conceito de Entropia para medir a "desordem" ou incerteza dos dados.

A cada nível da árvore, o ID3 escolhe a pergunta (ou posição do tabuleiro) que melhor separa as jogadas vitoriosas das perdedoras, maximizando o ganho de informação.

Tem o objectivo de transformar milhares de exemplos de jogadas complexas num conjunto de regras lógicas ("Se a casa X estiver ocupada, então joga na coluna Y") que podem ser seguidas de forma imediata.

ID3 Engine:

Algoritmo baseado em Entropia e Ganho de Informação.

Construção de uma hierarquia de decisões lógica.

Foco em reduzir a incerteza na escolha da jogada.

In [5]:
import numpy as np
from collections import Counter
import math

class DecisionNode:
    """Representa um nó ou folha da árvore."""
    def __init__(self, left=None, right=None, feature_idx=None, threshold=None, class_label=None):
        self.left = left
        self.right = right
        self.feature_idx = feature_idx  # Índice do atributo (ex: 0 para sepal length)
        self.threshold = threshold      # Valor de corte (v)
        self.class_label = class_label  # Previsão (apenas para folhas)

    def is_leaf(self):
        return self.class_label is not None

class ID3DecisionTree:
    def __init__(self, depth_limit=float('inf')):
        self.root = None
        self.depth_limit = depth_limit

    def _entropy(self, y):
        """Calcula a Entropia de um conjunto de etiquetas."""
        if len(y) == 0: return 0
        counts = Counter(y)
        probs = [counts[c] / len(y) for c in counts]
        return -sum(p * math.log2(p) for p in probs if p > 0)

    def _information_gain(self, y, y_left, y_right):
        """Calcula o ganho de informação de um split."""
        parent_ent = self._entropy(y)
        n = len(y)
        n_l, n_r = len(y_left), len(y_right)
        if n_l == 0 or n_r == 0: return 0
        
        child_ent = (n_l / n) * self._entropy(y_left) + (n_r / n) * self._entropy(y_right)
        return parent_ent - child_ent

    def _best_split(self, X, y):
        """Encontra o melhor atributo e o melhor ponto de corte (Discretização)."""
        best_gain = -1
        split_idx, split_thresh = None, None

        for idx in range(X.shape[1]):
            column_values = X[:, idx]
            # Ordenar valores únicos para testar pontos médios (Discretização otimizada)
            unique_values = np.unique(column_values)
            thresholds = (unique_values[:-1] + unique_values[1:]) / 2

            for t in thresholds:
                left_mask = column_values <= t
                y_l, y_r = y[left_mask], y[~left_mask]
                
                gain = self._information_gain(y, y_l, y_r)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = idx
                    split_thresh = t
        
        return split_idx, split_thresh

    def fit(self, X, y):
        """Treina a árvore."""
        self.root = self._build_tree(X, y)

    def _build_tree(self, X, y, depth=0):
        # Casos base
        if len(set(y)) == 1:
            return DecisionNode(class_label=list(y)[0])
        
        if depth >= self.depth_limit or len(y) <= 1:
            most_common = Counter(y).most_common(1)[0][0]
            return DecisionNode(class_label=most_common)

        # Encontrar o melhor split (Discretização numérica)
        idx, thresh = self._best_split(X, y)
        
        if idx is None: # Se não houver ganho possível
            return DecisionNode(class_label=Counter(y).most_common(1)[0][0])

        # Criar ramos
        left_mask = X[:, idx] <= thresh
        left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right = self._build_tree(X[~left_mask], y[~left_mask], depth + 1)
        
        return DecisionNode(left, right, idx, thresh)

    def predict(self, X):
        """Classifica novos exemplos."""
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.is_leaf():
            return node.class_label
        
        if x[node.feature_idx] <= node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)

    def show(self, node=None, depth=0, feature_names=None):
        """Apresenta a árvore visualmente."""
        if node is None: node = self.root
        
        indent = "  " * depth
        if node.is_leaf():
            print(f"{indent}PREVISÃO: {node.class_label}")
            return

        feat = feature_names[node.feature_idx] if feature_names else f"Attr {node.feature_idx}"
        print(f"{indent}[{feat} <= {node.threshold:.2f}]")
        self.show(node.left, depth + 1, feature_names)
        
        print(f"{indent}[{feat} > {node.threshold:.2f}]")
        self.show(node.right, depth + 1, feature_names)

# --- Exemplo de Uso com Carregamento Manual (CSV Iris) ---

def load_iris(file_path):
    # Carregamento simples sem pandas (usando apenas numpy/csv)
    data = np.genfromtxt(file_path, delimiter=',', dtype=None, encoding='utf-8')
    # Assumindo que a última coluna é a classe e as primeiras são atributos
    X = np.array([list(row)[:-1] for row in data], dtype=float)
    y = np.array([row[-1] for row in data])
    return X, y

# Exemplo de fluxo:
# X_train, y_train = load_iris('iris.data')
# tree = ID3DecisionTree(depth_limit=5)
# tree.fit(X_train, y_train)
# tree.show(feature_names=['sepal_len', 'sepal_wid', 'petal_len', 'petal_wid'])

In [ ]:
import numpy as np
import pandas as pd
from id3_engine import ID3DecisionTree # Importa a tua classe

def correr_projeto_iris():
    print("--- A carregar dados do Iris ---")
    
    # 1. Carregar o CSV
    try:
        df = pd.read_csv('iris.csv')
    except FileNotFoundError:
        print("Erro: O ficheiro iris.csv não foi encontrado na pasta!")
        return

    # 2. Pré-processamento (Obrigatório para este dataset)
    # O CSV que enviaste tem: [ID, sepallength, sepalwidth, petallength, petalwidth, class]
    # Precisamos de remover a coluna 'ID' porque ela baralha a árvore (o ID não ajuda a prever a flor)
    
    # Atributos (X): colunas da 1 à 4 (sepallength até petalwidth)
    X = df.iloc[:, 1:5].values
    
    # Etiquetas/Classes (y): última coluna (class)
    y = df.iloc[:, 5].values
    
    # Nomes dos atributos para a visualização
    feature_names = df.columns[1:5].tolist()

    # 3. Treinar a Árvore
    # Criamos a árvore com um limite de profundidade (ex: 4) para evitar sobreajuste
    tree = ID3DecisionTree(depth_limit=4)
    tree.fit(X, y)

    # 4. Mostrar a Árvore Visualmente (Requisito do enunciado)
    print("\n--- Estrutura da Árvore Gerada ---")
    tree.show(feature_names=feature_names)

    # 5. Testar com um exemplo novo (Aceitar test examples)
    print("\n--- Teste de Classificação ---")
    # Vamos inventar uma flor: SepalL=5.1, SepalW=3.5, PetalL=1.4, PetalW=0.2
    exemplo = np.array([[5.1, 3.5, 1.4, 0.2]])
    resultado = tree.predict(exemplo)
    print(f"Dados inseridos: {exemplo}")
    print(f"Resultado da Árvore: {resultado[0]}")

if __name__ == "__main__":
    correr_projeto_iris()

In [ ]:
import numpy as np
import pandas as pd
from id3_engine import ID3DecisionTree

def preparar_dados_e_treinar(nome_ficheiro):
    print(f"--- A carregar dataset: {nome_ficheiro} ---")
    try:
        df = pd.read_csv(nome_ficheiro)
        
        # O teu CSV tem 45 colunas. 
        # Vamos usar as primeiras 43 (tabuleiro + turno) como atributos (X)
        X = df.iloc[:, 0:43].values
        
        # A coluna 'chosen_move' (índice 43) é o que queremos prever (y)
        y = df.iloc[:, 43].values
        
        print(f"A treinar ID3 com {len(X)} exemplos de jogadas...")
        # Aumentamos a profundidade para 20 para captar melhor a estratégia do jogo
        tree = ID3DecisionTree(depth_limit=20) 
        tree.fit(X, y)
        
        # Teste de Accuracy rápido
        previsoes = tree.predict(X)
        acc = np.mean(previsoes == y) * 100
        print(f"Treino concluído! Accuracy no set de treino: {acc:.2f}%")
        
        return tree
    except Exception as e:
        print(f"Erro ao carregar ou treinar: {e}")
        return None

if __name__ == "__main__":
    # NOME EXATO DO TEU FICHEIRO
    nome_do_csv = 'mcts_2min_vs_random.csv' 
    
    arvore = preparar_dados_e_treinar(nome_do_csv)
    
    if arvore:
        print("\n[SUCESSO] A árvore está pronta para ser usada no play_game.py!")
    else:
        print("\n[ERRO] Verifica se o ficheiro mcts_2min_vs_random.csv está na mesma pasta.")

## ID3 Play

O ID3 Play é a classe que atua como a interface entre o algoritmo de árvore de decisão e o motor do jogo PopOut. É o "cérebro" do bot em ação.

Na fase de Treino e Validação: Antes de jogar, o ID3 Play carrega os dados e realiza testes rigorosos (Holdout 70/30 e Cross-Validation). Isto garante que o bot não apenas decorou as jogadas, mas que realmente aprendeu a lógica do jogo e consegue lidar com situações novas.

Tradução de Dados: O motor do jogo utiliza Bitboards (matemática binária rápida), mas a árvore de decisão precisa de uma visão clara do tabuleiro. O ID3 Play faz a ponte, traduzindo o estado binário para um formato que a árvore consegue interpretar (-1, 0, 1).

Eficiência: Ao contrário do MCTS, que precisa de simular milhares de jogos para decidir, o ID3 Play consulta a árvore e toma uma decisão instantânea, tornando-o um adversário extremamente rápido e leve em termos de processamento.

ID3 Play:

Integração direta com o motor binário do Connect4.

Aplicação de K-Fold Cross-Validation para garantir robustez.

Execução em tempo real com baixo custo computacional.

In [ ]:
from id3_engine import ID3DecisionTree
import numpy as np
import pandas as pd
import random

# --- NOVA CLASSE AGENTE ID3 COM VALIDAÇÃO CRUZADA ---

class ID3Play:
    def __init__(self, name="ID3-Bot", csv_path='mcts_2min_vs_random.csv'):
        self.name = name
        self.tree = ID3DecisionTree(depth_limit=20)
        self.csv_path = csv_path
        self._prepare_model()

    def _prepare_model(self):
        print(f"\n[!] A treinar e validar o {self.name}...")
        try:
            df = pd.read_csv(self.csv_path)
            X = df.iloc[:, 0:43].values
            y = df.iloc[:, 43].values
            
            # --- MELHORIA 1: 70% TREINO / 30% TESTE ---
            indices = np.random.permutation(len(X))
            train_idx = int(0.7 * len(X))
            X_train, X_test = X[indices[:train_idx]], X[indices[train_idx:]]
            y_train, y_test = y[indices[:train_idx]], y[indices[train_idx:]]
            
            self.tree.fit(X_train, y_train)
            preds = self.tree.predict(X_test)
            acc_holdout = np.mean(preds == y_test) * 100
            print(f"-> Sucesso: Accuracy (Holdout 70/30): {acc_holdout:.2f}%")

            # --- MELHORIA 2: CROSS-VALIDATION (K=5) ---
            print("-> A executar 5-Fold Cross-Validation...")
            k = 5
            fold_size = len(X) // k
            cv_scores = []
            for i in range(k):
                s, e = i * fold_size, (i + 1) * fold_size
                X_val_f, y_val_f = X[s:e], y[s:e]
                X_train_f = np.concatenate([X[:s], X[e:]])
                y_train_f = np.concatenate([y[:s], y[e:]])
                
                temp_tree = ID3DecisionTree(depth_limit=15)
                temp_tree.fit(X_train_f, y_train_f)
                cv_scores.append(np.mean(temp_tree.predict(X_val_f) == y_val_f))
            
            print(f"-> Média Cross-Validation: {np.mean(cv_scores)*100:.2f}% (+/- {np.std(cv_scores)*100:.2f}%)")
            
        except Exception as e:
            print(f"Erro ao carregar dados: {e}")

    def get_move(self, state):
        # Converter bits do motor para formato do CSV (-1, 0, 1)
        board_array = np.full(42, -1)
        p0 = state.position if state.turn == 0 else (state.position ^ state.mask)
        p1 = state.position if state.turn == 1 else (state.position ^ state.mask)
        for i in range(42):
            bit = 1 << ((i % 7) * 7 + (i // 7))
            if p0 & bit: board_array[i] = 0
            elif p1 & bit: board_array[i] = 1
        
        input_v = np.append(board_array, state.turn)
        move = int(self.tree.predict(np.array([input_v]))[0])
        
        # Validação de segurança
        legal = state.legal_moves()
        return move if move in legal else random.choice(legal)

## Game Loop

In [7]:
"""

# Game loop
if __name__ == "__main__":
    
    def choose_player(num_player):
        while True:
            print(f"\nChoose a player for {num_player}")
            print("1 - Human")
            print("2 - Random Play")
            print("3 - MCTS")
            print("4 - ID3 (MCTS)")
            player_str = input("Enter 1 or 2 or 3 or 4: ").strip()
            
            if player_str == '1':
                return HumanPlay(f"Player {num_player} (Human)")
            elif player_str == '2':
                return RandomPlay(f"Player {num_player} (Random)")
            elif player_str == '3':
                return MCTSPlay(f"Player {num_player} (MCTS)", time_limit=2.0)
            elif player_str == '4':
                return ID3Play(f"Player {num_player} (ID3)")
            else:
                print("Invalid player! Please enter 1, 2 or 3.")

    agents = {
        0: choose_player(0),
        1: choose_player(1)
    }

    game = Connect4()
    pos = game.get_initial_position()
    
    print(f"\n--- Game start: {agents[0].name} VS {agents[1].name} ---")

    # memory stored to check for draws
    global_history = {}

    # Main Loop
    while not pos.terminal:
        # updates the current board state occurences
        global_history[pos.hash] = global_history.get(pos.hash, 0) + 1
        
        legal = pos.legal_moves()
        
        # Injects the draw option (-1) if threefold repetition is met
        if global_history[pos.hash] >= 3 and -1 not in legal:
            legal.append(-1)
            print("\n[!] Attention: This state has repeated 3 times. Draw (-1) is available.")

        pos.print_board(legal_moves=legal)
        
        current_agent = agents[pos.turn]
        print(f"\n{current_agent.name}'s turn.")
        
        move = current_agent.get_move(pos)
        pos = pos.move(move)

    # End of game
    pos.print_board()
    if pos.result == 0:
        print("It's a Draw!")
    else:
        winner = 0 if pos.result == 1 else 1
        print(f"Player {winner} wins!")

"""


Choose a player for 0
1 - Human
2 - Random Play
3 - MCTS
4 - ID3 (MCTS)

Choose a player for 1
1 - Human
2 - Random Play
3 - MCTS
4 - ID3 (MCTS)

--- Game start: Player 0 (Random) VS Player 1 (MCTS) ---

  7  8  9 10 11 12 13 : Pop
  0  1  2  3  4  5  6 : Drop
-----------------------
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
-----------------------
Legal Moves:
  Drops: [0, 1, 2, 3, 4, 5, 6]


Player 0 (Random)'s turn.
Player 0 (Random) played: 1

  7  8  9 10 11 12 13 : Pop
  0  1  2  3  4  5  6 : Drop
-----------------------
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  .  .  .  .  .  . |
| .  O  .  .  .  .  . |
-----------------------
Legal Moves:
  Drops: [0, 1, 2, 3, 4, 5, 6]


Player 1 (MCTS)'s turn.
[Player 1 (MCTS)] Thinking for 2.0 seconds using 4 cores...
